- **Runtime :** 14.3 LTS ML (includes Apache Spark 3.5.0, GPU, Scala 2.12) 
- **Driver :** g5.4xlarge [A10G]
- **Librairies**: 
  - poetry==1.5.1

In [0]:
%sh
pip install .

In [0]:
%pip install --force-reinstall --no-deps  neuralforecast-1.7.5-qkcv.tar

dbutils.library.restartPython()

In [0]:
import time, sys, os 
import datetime as dt
print(dt.datetime.now().strftime('%Y%m%d%H%M'))
print('#'*20 + dt.datetime.now().strftime(' %Y.%m.%d ') + '#'*20)

sys.path.append(os.path.abspath("static_feature_in_attention_layer"))
from DB_common_utils import *

dt_current = dt.datetime.now().strftime('%Y%m%d')
dt_current


## <font color=red>load data:  </font>   
<font color=yellow>Retail:  </font> https://www.kaggle.com/c/favorita-grocery-sales-forecasting/   




### <font color=red>Dataset: Sales</font> 

In [0]:
import s3fs,os
fs = s3fs.S3FileSystem()

In [0]:
_use_staging_train = True # skip data proceeding
_save_to_staging = True # update staging file
_no_training = False # skip train
_no_tuning = True # skip tunning
_log_transform = False # if use log

_features_static = [ 'city', 'state', 'type_store', 'cluster'] + ['family', 'class', 'perishable']
_features_dynamic = ['type', 'locale', 'locale_name','transferred']

In [0]:
### reader
import pyarrow.parquet as pq
import pandas as pd
_p_base = 's3://'

_p_base_retail = os.path.join(_p_base, 'favorita-grocery-sales-forecasting')

In [0]:
_df_items = pd.read_csv(os.path.join(_p_base_retail,'items.csv'))
_df_stores = pd.read_csv(os.path.join(_p_base_retail,'stores.csv'))
_df_stores.rename(columns={'type': 'type_store'}, inplace=True)

_df_oil = pd.read_csv(os.path.join(_p_base_retail,'oil.csv'))
_df_holidays_events = pd.read_csv(os.path.join(_p_base_retail,'holidays_events.csv'))
_df_holidays_events

,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False
...,...,...,...,...,...,...
345,2017-12-22,Additional,National,Ecuador,Navidad-3,False
346,2017-12-23,Additional,National,Ecuador,Navidad-2,False
347,2017-12-24,Additional,National,Ecuador,Navidad-1,False
348,2017-12-25,Holiday,National,Ecuador,Navidad,False


In [0]:
import glob

if not _use_staging_train:
    _df_train = pd.read_csv(os.path.join(_p_base_retail, 'train.csv'), sep=',')

    _df_train = _df_train.loc[(_df_train.date >= '0015-01-01') & (_df_train.date < '2016-03-01')]
    gc.collect()

    print(f"date.min {_df_train.date.min()}, date.max {_df_train.date.max()}, item_nbr.max {_df_train.item_nbr.max()}, unit_sales.min {_df_train.unit_sales.min()}, unit_sales.max() {_df_train.unit_sales.max()}")

    unique_id = 'siid'
    _df_train[unique_id] = _df_train['store_nbr']*100000000 + _df_train['item_nbr']

    df_cross_join = _df_train[['date']].drop_duplicates().assign(key=1).merge(_df_train[[unique_id, 'store_nbr', 'item_nbr']].drop_duplicates().assign(key=1), on='key').drop('key', axis=1)

    _df_train['open_flag'] = 1
    _df_train = df_cross_join.merge(_df_train, on=['date', unique_id, 'store_nbr', 'item_nbr'], how='left').sort_values([unique_id, 'date']).reset_index()

    _df_train.count()
    _df_train.isnull().sum()

    _df_train['unit_sales'] = _df_train.groupby(unique_id)['unit_sales'].ffill()
    _df_train = _df_train.loc[(_df_train.date >= '2015-01-01') & (_df_train.date < '2016-03-01')]

    _df_train.isnull().sum()

    _df_train['unit_sales'].fillna(0, inplace=True)
    _df_train['open_flag'].fillna(0, inplace=True)

    _df_complete_numeric = _df_train.merge(_df_oil, on='date', how='left').merge(_df_holidays_events, on='date', how='left')

    _df_complete_numeric.rename(columns={unique_id: 'unique_id',
                                'unit_sales': 'y',
                                'date': 'ds'}, inplace=True)
    _df_complete_numeric["ds"] = pd.to_datetime(_df_complete_numeric["ds"])

    _df_train = None
    gc.collect()

    for _c in _features_dynamic:
        _df_complete_numeric[_c] = _df_complete_numeric[_c].astype('category').cat.codes
    _dt_max = _df_complete_numeric.ds.max()
    _df_complete_numeric = _df_complete_numeric.loc[_df_complete_numeric.item_nbr.isin(_df_complete_numeric.loc[_df_complete_numeric.ds == _dt_max, 'item_nbr'])].reset_index(drop=True)

    _df_complete_numeric.y.fillna(0.0001, inplace=True)

    if _save_to_staging:

        print(f"saving files, y max {_df_complete_numeric['y'].max()}, min {_df_complete_numeric['y'].min()}")
        _df_complete_numeric.ds.max()

        ### save _df_complete_numeric
        num_splits = 10

        # Split the dataframe into smaller dataframes
        df_splits = np.array_split(_df_complete_numeric, num_splits)

        # Save each split into a separate parquet file
        for i, df_split in enumerate(df_splits):
            df_split.to_parquet(os.path.join(_p_base_retail, f'_df_complete_numeric_part_{i}.parquet'))

else:
    # Read all parquet files in the specified directory
    parquet_files =[]
    for i in range(10):
        parquet_files.append(os.path.join(_p_base_retail, f'_df_complete_numeric_part_{i}.parquet'))
    print(f'parquet_files {parquet_files}')
    # Concatenate all the parquet files into a single dataframe
    _df_complete_numeric = pd.concat([pd.read_parquet(file) for file in parquet_files])

_df_complete_numeric.describe()


In [0]:
### _df_static
_df_complete_numeric.columns
_df_items.columns

_df_static = _df_complete_numeric[['unique_id','store_nbr','item_nbr']].drop_duplicates().merge(_df_stores, on='store_nbr', how='left').merge(_df_items, on='item_nbr', how='left')

_df_static.columns

_df_static_numeric = _df_static.copy()
for _c in _features_static:
    _df_static_numeric[_c] = _df_static_numeric[_c].astype('category').cat.codes

_df_static_numeric

In [0]:
gc.collect()
if _no_training:
    raise ValueError(f"Paused before training.")

189

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from neuralforecast import NeuralForecast
from neuralforecast.models import TFT_v1
from neuralforecast.losses.pytorch import DistributionLoss,MAE
from neuralforecast.utils import AirPassengersPanel, AirPassengersStatic

if _log_transform:
    _df_complete_numeric['y'] = np.log(_df_complete_numeric['y'])


Y_train_df = _df_complete_numeric.loc[(_df_complete_numeric.ds >= '2015-01-01') & (_df_complete_numeric.ds < '2015-12-01'), _features_dynamic + ['unique_id', 'ds', 'y']]


In [0]:

nf = NeuralForecast(
    models=[TFT_v1(h=30, input_size=60,
                hidden_size=64,#128,
                n_head=2,
                # grn_activation='ELU',
                loss=DistributionLoss(distribution='StudentT', level=[10, 50, 80, 90]), #MAE(), #DistributionLoss(distribution='StudentT', level=[80, 90]),
                learning_rate=0.01,
                batch_size=256,
                dropout=0.2,
                stat_exog_list=_features_static,
                futr_exog_list=_features_dynamic,
                hist_exog_list=[],
                max_steps=5000,
                val_check_steps=5000,
                early_stop_patience_steps=10,
                scaler_type='robust',
                windows_batch_size=1024,
                enable_progress_bar=True,
                v_qkcv=0,
                # lstm_layer=1,
                ),
            TFT_v1(h=30, input_size=60,
                hidden_size=64,#128,
                n_head=2,
                # grn_activation='ELU',
                loss=DistributionLoss(distribution='StudentT', level=[10, 50, 80, 90]), #MAE(), #DistributionLoss(distribution='StudentT', level=[80, 90]),
                learning_rate=0.01,
                batch_size=256,
                dropout=0.2,
                stat_exog_list=_features_static,
                futr_exog_list=_features_dynamic,
                hist_exog_list=[],
                max_steps=5000,
                val_check_steps=5000,
                early_stop_patience_steps=10,
                scaler_type='robust',
                windows_batch_size=1024,
                enable_progress_bar=True,
                v_qkcv=1,
                # lstm_layer=1,
                ),
            TFT_v1(h=30, input_size=60,
                hidden_size=64,#128,
                n_head=2,
                # grn_activation='ELU',
                loss=DistributionLoss(distribution='StudentT', level=[10, 50, 80, 90]), #MAE(), #DistributionLoss(distribution='StudentT', level=[80, 90]),
                learning_rate=0.01,
                batch_size=256,
                dropout=0.2,
                stat_exog_list=_features_static,
                futr_exog_list=_features_dynamic,
                hist_exog_list=[],
                max_steps=5000,
                val_check_steps=5000,
                early_stop_patience_steps=10,
                scaler_type='robust',
                windows_batch_size=1024,
                enable_progress_bar=True,
                v_qkcv=2,
                # lstm_layer=1,
                ),
            TFT_v1(h=30, input_size=60,
                hidden_size=64,#128,
                n_head=2,
                # grn_activation='ELU',
                loss=DistributionLoss(distribution='StudentT', level=[10, 50, 80, 90]), #MAE(), #DistributionLoss(distribution='StudentT', level=[80, 90]),
                learning_rate=0.01,
                batch_size=256,
                dropout=0.2,
                stat_exog_list=_features_static,
                futr_exog_list=_features_dynamic,
                hist_exog_list=[],
                max_steps=5000,
                val_check_steps=5000,
                early_stop_patience_steps=10,
                scaler_type='robust',
                windows_batch_size=1024,
                enable_progress_bar=True,
                v_qkcv=3,
                # lstm_layer=1,
                ),
    ],
    freq='D'
)
nf.fit(df=Y_train_df, static_df=_df_static_numeric, val_size=30)


In [0]:
_col_forecast='TFT_v1'

Y_test_df = _df_complete_numeric.loc[(_df_complete_numeric.ds >= '2016-01-01') & (_df_complete_numeric.ds < '2016-02-01'), _features_dynamic + ['unique_id', 'ds', 'y']]

Y_hat_df = nf.predict(futr_df=Y_test_df
                      , df=_df_complete_numeric.loc[(_df_complete_numeric.ds >= '2015-01-01') & (_df_complete_numeric.ds < '2016-01-01'), _features_dynamic + ['unique_id', 'ds', 'y']]
                      , static_df=_df_static_numeric)

if _log_transform:
    Y_hat_df[_col_forecast] = np.exp(Y_hat_df[_col_forecast]).round().astype(int)
    Y_test_df['y'] = np.exp(Y_test_df['y']).round().astype(int)

In [0]:
print(f'Y_hat_df {Y_hat_df.columns} {Y_hat_df.ds.min()} {Y_hat_df.ds.max()}')

_df_meged = Y_hat_df.merge(Y_test_df, on=['ds','unique_id'], how='left').fillna(0)

_df_meged['forecast_step'] = _df_meged.sort_values('ds').groupby('unique_id').cumcount() + 1
for _c in [f'{_col_forecast}', f'{_col_forecast}-median', f'{_col_forecast}-lo-90', f'{_col_forecast}-lo-80', f'{_col_forecast}-lo-50', f'{_col_forecast}-hi-50', f'{_col_forecast}-hi-80', f'{_col_forecast}-hi-90']:
    _df_meged[_c]=_df_meged[_c].round().astype(int)
_df_meged


In [0]:

def wpe_func(forecast_base_ori, eval_horizon=[4, 12, 52], real='y', forecast='TFT-median'):
    objective_metric = 0
    forecast_base = forecast_base_ori.dropna()
    for horizon in eval_horizon:

        wpe = (
            forecast_base[
                (forecast_base.forecast_step <= horizon)
            ]
            .groupby(by=["unique_id"], as_index=False)
            .agg({real: "sum", forecast: "sum"})
        )

        wpe["gap_qty"] = abs(wpe[real] - wpe[forecast])
        wpe["wpe_{}W_qty".format(horizon)] = wpe["gap_qty"] / wpe[real]

        wpe_all = wpe.agg(
            {"gap_qty": "sum", real: "sum",}
        )
        wpe_all["wpe_{}W_qty".format(horizon)] = (
            wpe_all["gap_qty"] / wpe_all[real]
        )

        print(wpe_all["gap_qty"] / wpe_all[real])

    return wpe, wpe_all



In [0]:
_op2 = wpe_func(_df_meged, eval_horizon=[30], real='y', forecast=f'{_col_forecast}-median')

print(_op2[0])
print(_op2[1])

0.22372160064209745
         unique_id      y  TFT_v1-median  gap_qty  wpe_30W_qty
0        100096995   30.0             30      0.0     0.000000
1        100099197   30.0             30      0.0     0.000000
2        100103520   56.0             87     31.0     0.553571
3        100103665  104.0             92     12.0     0.115385
4        100105574  139.0            113     26.0     0.187050
...            ...    ...            ...      ...          ...
143370  5402026893   81.0             97     16.0     0.197531
143371  5402026945  180.0             93     87.0     0.483333
143372  5402026983  110.0            251    141.0     1.281818
143373  5402027090   48.0             62     14.0     0.291667
143374  5402027252  149.0            161     12.0     0.080537

[143375 rows x 5 columns]
gap_qty        6.401842e+06
y              2.861521e+07
wpe_30W_qty    2.237216e-01
dtype: float64


In [0]:
def quantile_loss(y_true, y_pred, quantile):
    error = y_true - y_pred
    # print(np.maximum(quantile * error, 0))
    return 2* (
        quantile * np.maximum(error, 0)+ (1 - quantile) * np.maximum(-error, 0)).mean() / np.mean(np.abs(y_true))
    
def calculate_quantile_losses(df_forecast, df_real, real_col='y'):
    df_merged = df_forecast.merge(df_real, on=['ds', 'unique_id'], how='left').fillna(0)
    p50_loss = quantile_loss(df_merged[real_col], df_merged[f'{_col_forecast}-median'], 0.5)
    p90_loss_lo = quantile_loss(df_merged[real_col], df_merged[f'{_col_forecast}-lo-90'], 0.9)
    p90_loss_hi = quantile_loss(df_merged[real_col], df_merged[f'{_col_forecast}-hi-90'], 0.9)
    return p50_loss, p90_loss_lo, p90_loss_hi

# Example usage
p50_loss, p90_loss_lo, p90_loss_hi = calculate_quantile_losses(Y_hat_df, Y_test_df)
print(f"P50 Loss: {p50_loss}")
print(f"P90 Loss lo: {p90_loss_lo}")
print(f"P90 Loss hi: {p90_loss_hi}")

P50 Loss: 0.4752658329107674
P90 Loss lo: 1.992147316126936
P90 Loss hi: 0.31758067798244366



## <font color=yellow>Analysis</font> 

In [0]:
# feature_importances = nf.models[0].feature_importances()
# feature_importances.keys()

_self = nf.models[3]
static_encoder_sparse_weights = _self.interpretability_params.get(
                "static_encoder_sparse_weights"
            )

static_encoder_sparse_weights[0]

static_encoder_sparse_weights[0].size()

static_vsn_imp = pd.DataFrame(
                _self.mean_on_batch(static_encoder_sparse_weights[0]).cpu().numpy(),
                index=_self.stat_exog_list,
                columns=["importance"],
            )

static_vsn_imp

tensor([[0.0180, 0.0658, 0.5592, 0.0451, 0.1775, 0.1061, 0.0284],
        [0.0179, 0.0652, 0.5606, 0.0458, 0.1763, 0.1058, 0.0284],
        [0.0179, 0.0652, 0.5606, 0.0458, 0.1763, 0.1058, 0.0284],
        [0.0179, 0.0652, 0.5606, 0.0458, 0.1763, 0.1058, 0.0284],
        [0.0179, 0.0652, 0.5606, 0.0458, 0.1763, 0.1058, 0.0284],
        [0.0179, 0.0655, 0.5599, 0.0455, 0.1767, 0.1061, 0.0284],
        [0.0179, 0.0652, 0.5606, 0.0458, 0.1763, 0.1058, 0.0284],
        [0.0179, 0.0652, 0.5606, 0.0458, 0.1763, 0.1058, 0.0284],
        [0.0179, 0.0655, 0.5599, 0.0455, 0.1767, 0.1061, 0.0284],
        [0.0179, 0.0652, 0.5606, 0.0458, 0.1763, 0.1058, 0.0284],
        [0.0179, 0.0652, 0.5606, 0.0458, 0.1763, 0.1058, 0.0284],
        [0.0179, 0.0652, 0.5606, 0.0458, 0.1763, 0.1058, 0.0284],
        [0.0179, 0.0652, 0.5606, 0.0458, 0.1763, 0.1058, 0.0284],
        [0.0179, 0.0655, 0.5599, 0.0455, 0.1767, 0.1061, 0.0284],
        [0.0179, 0.0652, 0.5606, 0.0458, 0.1763, 0.1058, 0.0284]],
       de

torch.Size([15, 7])

,importance
city,0.017906
state,0.065301
type_store,0.560331
cluster,0.045714
family,0.176447
class,0.105881
perishable,0.028421


## (Optional) <font color=yellow>Tuning</font> 

In [0]:
if _no_tuning:
    raise ValueError(f"Paused before tuning.")

In [0]:
from hyperopt import hp, fmin, tpe, Trials, space_eval

eval_horizon=[4,10,30]

def calculate_wape_ori(df_forecast, df_actual, tag, log=False):
    df_error = pd.merge(df_forecast, df_actual, how="left").fillna(0)
    df_error = df_error[df_error["week_id"] <= df_actual["week_id"].max()]
    df_error["forecast_step"] = list(range(1, df_error.week_id.nunique() + 1)) * df_error["model_id"].nunique()
    objective_metric = 0
    for h in eval_horizon:
        tmp = df_error[df_error["forecast_step"] <= h]
        wape = np.round(np.sum(np.abs(tmp["sales_quantity"] - tmp["forecast"])) / np.sum(tmp["sales_quantity"]), 3)
        h_min = min(h, tmp["forecast_step"].max())
        if log:
            mlflow.log_metric(f"WAPE_{tag}_{h_min:02d}", wape)
        if h_min in [4, 12, 52]:
            objective_metric += wape
        if h >= df_error["forecast_step"].max():
            break
    return objective_metric

In [0]:
horizon = 30
space = {
    "input_size": hp.choice("input_size", [2*horizon, 3*horizon]), 
    "hidden_size": hp.choice("hidden_size", [32, 64, 128, 256]),
    "n_head": hp.choice("n_head", [1, 2, 4]),
    "learning_rate": hp.choice("learning_rate", [0.01,0.001,0.0001]), 
    "dropout": hp.choice("dropout", [0.1, 0.2, 0.3]),
    "scaler_type": hp.choice("scaler_type", ["robust"]),
    "max_steps": hp.choice("max_steps", [1000, 3000, 5000]),
    "batch_size": hp.choice("batch_size", [64, 128, 256, 512]),
    "windows_batch_size": hp.choice("windows_batch_size", [256, 512, 1024]),

}

In [0]:
random_seed = 42

Y_valid_df = _df_complete_numeric.loc[(_df_complete_numeric.ds >= '2015-12-01') & (_df_complete_numeric.ds < '2016-01-01'), _features_dynamic + ['unique_id', 'ds', 'y']]
print(f'Y_valid_df {Y_valid_df.ds.min()} {Y_valid_df.ds.max()}')

def calculate_wape_metric(df_error, eval_horizon=[4, 12, 52], tag='', log=False):
    print(f"calc WAPE_{tag}, df_error contains {df_error.columns}")
    objective_metric = 0
    for h in eval_horizon:
        tmp = df_error[df_error["forecast_step"] <= h]
        wape = np.round(np.sum(np.abs(tmp["y"] - tmp["TFT"])) / np.sum(tmp["y"]), 5)
        h_min = min(h, tmp["forecast_step"].max())
        
        print(f"calc WAPE_{tag}_{h_min:02d} wape {wape}")
        if h_min in [4, 12, 52]:
            objective_metric += wape
        if h >= df_error["forecast_step"].max():
            print(f"h = {h}, while max step is {df_error.forecast_step.max()}")
            break
    return objective_metric

def calculate_quantile_losses_metric(df_merged, eval_horizon=[30], real_col='y'):
    p50_loss = quantile_loss(df_merged[real_col], df_merged['TFT-median'], 0.5)
    # p90_loss = quantile_loss(df_merged[real_col], df_merged[forecast_col], 0.9)
    print(f"calc p50_loss {p50_loss}")
    return p50_loss

def fn(params): 

    input_size = int(params["input_size"])
    hidden_size = int(params["hidden_size"])
    n_head = int(params["n_head"])
    learning_rate = params["learning_rate"]
    dropout= params["dropout"]
    scaler_type = params["scaler_type"]
    max_steps = int(params["max_steps"])
    # loss = eval(params["loss"])
    batch_size = int(params["batch_size"]) #int(2**params["batch_size_exponent"])
    windows_batch_size = int(params["windows_batch_size"]) if params["windows_batch_size"] else None #int(2**params["windows_batch_size_exponent"])

    tft = TFT(
        h=horizon,
        input_size=input_size,
        tgt_size=1,
        futr_exog_list=_features_dynamic if len(_features_dynamic) > 0 else None,
        hist_exog_list=[],
        stat_exog_list=_features_static if len(_features_static) > 0 else None,
        hidden_size=hidden_size,
        n_head=n_head,
        attn_dropout=0.0, # dropout of fusion decoder's attention layer
        dropout=dropout,
        loss=DistributionLoss(distribution='StudentT', level=[10, 50, 80, 90]),
        valid_loss=None,
        max_steps=max_steps,
        learning_rate=learning_rate,
        num_lr_decays=-1,
        early_stop_patience_steps=-1,
        val_check_steps=max_steps+1,
        batch_size=batch_size,
        valid_batch_size=None,
        windows_batch_size=windows_batch_size,
        inference_windows_batch_size=windows_batch_size,
        step_size=1,
        scaler_type=scaler_type,
        num_workers_loader=0,
        drop_last_loader=False,
        random_seed=random_seed,
        deterministic="warn",
    )
        
    forecastor = NeuralForecast(models=[tft], freq='D')
    forecastor.fit(df=Y_train_df, static_df=_df_static_numeric, val_size=30)
    Y_hat_df = forecastor.predict(futr_df=Y_valid_df, df=Y_train_df, static_df=_df_static_numeric)

    _df_meged = Y_hat_df.merge(Y_valid_df, on=['ds','unique_id'], how='left').fillna(0)
    _df_meged['forecast_step'] = _df_meged.sort_values('ds').groupby('unique_id').cumcount() + 1
    for _c in ['TFT', 'TFT-median', 'TFT-lo-90', 'TFT-lo-50', 'TFT-hi-50', 'TFT-hi-90']:
        _df_meged[_c]=_df_meged[_c].round().astype(int)
    # objective_metric_rec = calculate_wape_metric(_df_meged, eval_horizon=[4, 30])

    return calculate_quantile_losses_metric(_df_meged)

In [0]:
max_evals = 50
timeout = None #3600 * 2
trials = Trials()

best = fmin(
    fn=fn, 
    space=space, # Ensure no duplicate keys in space
    algo=tpe.suggest, 
    max_evals=max_evals,
    timeout=timeout,
    trials=trials,
    rstate=np.random.default_rng(random_seed),
)

In [0]:
space_eval(space, best)

In [0]:
res_df = (
    pd.DataFrame({"tid" : trials.tids, "tloss" : trials.losses()})
    .join(pd.DataFrame([space_eval(space, {x[0]: x[1][0] for x in trial['misc']['vals'].items()}) for trial in trials.trials]))
)
res_df.display()